In [0]:

   


  -- 步骤1:     
    -- 步骤2: 创建增量数据临时视图
    -- 假设 silver.orders 有 created_at 或 modified_at 时间戳
    -- 查找上次处理后新增或修改的数据
    CREATE OR REPLACE TEMPORARY VIEW incremental_sales AS
    WITH 
    last_processed_time AS (
        SELECT 
            COALESCE(MAX(update_time), TIMESTAMP('1900-01-01')) as last_time
        FROM gold_fact_sales
    )
    select
        od.id,
        od.order_id,
        user_id,
        sku_id,
        province_id,
        to_number(to_char(od.create_time, 'yyyyMMdd'),99999999) as date_id,
        od.create_time,
        sku_num,
        order_price,
        sku_num * order_price split_original_amount,
        nvl(split_activity_amount,0.0) split_activity_amount,
        nvl(split_coupon_amount,0.0) split_coupon_amount,
        split_total_amount,
        case when od.update_timestamp < oi.update_timestamp then oi.update_timestamp else od.update_timestamp end as ods_max_update_time
    from
        adhyivy.default.silver_order_detail od
        left join adhyivy.default.silver_orders_info oi on od.order_id = oi.id
        CROSS JOIN last_processed_time lpt
            WHERE od.update_timestamp > lpt.last_time or oi.update_timestamp > lpt.last_time  -- 增量条件,任一个表有更新的记录
    ;
    
    MERGE INTO gold_fact_sales AS target
    USING incremental_sales AS source
    ON target.id = source.id
    WHEN MATCHED THEN--AND target.update_timestamp < source.update_timestamp THEN
        UPDATE SET 
            target.order_id = source.order_id,
            target.user_id = source.user_id,
            target.sku_id = source.sku_id,
            target.province_id = source.province_id,
            target.date_id = source.date_id,
            target.create_time = source.create_time,
            target.sku_num = source.sku_num,
            target.order_price = source.order_price,
            target.split_original_amount = source.split_original_amount,
            target.split_activity_amount = source.split_activity_amount,
            target.split_coupon_amount = source.split_coupon_amount,
            target.split_total_amount = source.split_total_amount,
            target.ods_max_update_time = source.ods_max_update_time,
            target.load_time = source.load_time,
            target.update_time = CURRENT_TIMESTAMP()
    WHEN NOT MATCHED THEN
        INSERT (
            id,order_id, user_id, sku_id,
            province_id, date_id,
            create_time, sku_num, order_price, split_original_amount,
            split_activity_amount, split_coupon_amount, split_total_amount, ods_max_update_time,
            load_time, update_time
        )
        VALUES (
            source.id,source.order_id, source.user_id, source.sku_id,
            source.province_id, source.date_id,
            source.create_time, source.sku_num, source.order_price, source.split_original_amount,
            source.split_activity_amount, source.split_coupon_amount, source.split_total_amount, source.ods_max_update_time,
            CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()
        );